# Overfit ACS 2018 target and four-feature membership attack
Training size and settings are defined below; the existing target configuration is preserved. Target training uses a fixed epoch budget without early stopping or best-checkpoint restoration. Preprocessing fits only on actual target training records. Designated MIA members stay in training; held-out target records never enter training.

The restored attack matches experiments_1.ipynb: five diverse Adult shadows with their original default settings, class-scaled logit, blind walk, entropy, cross-entropy, and a fixed random forest attack. Auxiliary validation alone selects thresholds. This is the earlier exploratory configuration, not a guarantee of reproducing a particular accuracy. The original probability-only attack was rerun as the comparison baseline.

## Attack repair experiment
The original Adult probability-only attack remains a baseline. The additional
attacks use 16 ACS2018 reference models trained solely on unused source records,
disjoint from all target training and evaluation records. Reference models use
the target architecture and 100-epoch training budget, but a smaller training
size; measured gaps are reported rather than assumed to match.
Stable pre-softmax margins require additional logit access. Binary modified
entropy and exactly balanced grouped attack splits are included.
OUT-reference calibration is LiRA-inspired, not an exact online LiRA replication.
All hyperparameters are fixed in advance; thresholds use auxiliary validation only.
Source: https://arxiv.org/abs/2112.03570
The user authorized this run; future runs require approval.


In [1]:
from pathlib import Path
import json
import numpy as np
import torch
torch.set_num_threads(4)
import importlib
import pandas as pd
import src.data
import src.training
importlib.reload(src.data)
importlib.reload(src.training)
from src.data import load_acs_income, split_acs_data, subset_target_training, preprocess_partitions
from src.training import train_target_model, evaluate_model

OVERFIT_SEED = 42
OVERFIT_TRAIN_SIZE = 145_000
SPLIT_CONFIG = dict(validation_size=10_000, test_size=10_000,
                   mia_nonmember_size=10_000, mia_member_size=10_000, seed=OVERFIT_SEED)
OVERFIT_CONFIG = dict(
    hidden_dims=(512, 256, 128), dropout=0.0, epochs=100,
    batch_size=256, learning_rate=1e-3, weight_decay=0.0,
    seed=OVERFIT_SEED, device='cpu', early_stopping=False,
    restore_best=False, monitor_train=True,
)
REPAIR_DIR = Path('artifacts/exp2_attack_repair')
REPAIR_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
overfit_source = load_acs_income('data/processed/acs_income_ca_2018.csv')
baseline_partitions_for_ids = split_acs_data(overfit_source, **SPLIT_CONFIG)
overfit_partitions = subset_target_training(
    baseline_partitions_for_ids, train_size=OVERFIT_TRAIN_SIZE, seed=OVERFIT_SEED)
print('Unused baseline training records:',
      len(baseline_partitions_for_ids['train']) - len(overfit_partitions['train']))
pd.DataFrame({name: {'records': len(frame), 'positive_rate': frame['income_gt_50k'].mean()}
              for name, frame in overfit_partitions.items()}).T

Unused baseline training records: 20665


,records,positive_rate
validation,10000.0,0.410600
test,10000.0,0.410600
mia_nonmembers,10000.0,0.410600
train,145000.0,0.410572
mia_members,10000.0,0.410600


In [3]:
# Fit ONLY on the reduced training set. Do not reuse the baseline encoder.
overfit_preprocessor, overfit_arrays = preprocess_partitions(overfit_partitions)
{name: values['X'].shape for name, values in overfit_arrays.items()}

{'validation': (10000, 815),
 'test': (10000, 815),
 'mia_nonmembers': (10000, 815),
 'train': (145000, 815),
 'mia_members': (10000, 815)}

In [4]:
# This cell trains when YOU run it. A large CPU model may take time.
overfit_model, overfit_history, overfit_metadata = train_target_model(
    overfit_arrays['train']['X'], overfit_arrays['train']['y'],
    overfit_arrays['validation']['X'], overfit_arrays['validation']['y'],
    **OVERFIT_CONFIG)
print('Returned epoch:', overfit_metadata['returned_epoch'])
pd.DataFrame(overfit_history)
from src.training import save_target_model
import joblib
save_target_model(overfit_model, overfit_metadata, REPAIR_DIR / 'target_model.pt')
joblib.dump(overfit_preprocessor, REPAIR_DIR / 'target_preprocessor.joblib')


Epoch 001: train loss=0.3990, validation loss=0.3791, validation accuracy=0.8288


Epoch 002: train loss=0.3742, validation loss=0.3774, validation accuracy=0.8327


Epoch 003: train loss=0.3612, validation loss=0.3816, validation accuracy=0.8289


Epoch 004: train loss=0.3436, validation loss=0.3887, validation accuracy=0.8271


Epoch 005: train loss=0.3210, validation loss=0.3975, validation accuracy=0.8254


Epoch 006: train loss=0.2949, validation loss=0.4326, validation accuracy=0.8200


... (91 outputs left out)

Epoch 098: train loss=0.0294, validation loss=1.8926, validation accuracy=0.7901


Epoch 099: train loss=0.0283, validation loss=1.9240, validation accuracy=0.7909


Epoch 100: train loss=0.0269, validation loss=1.8905, validation accuracy=0.7890
Returned epoch: 100


['artifacts\\exp2_attack_repair\\target_preprocessor.joblib']

## Inspect overfitting
Compare end-of-epoch training accuracy/loss against validation. train_loss is averaged across updates during the epoch; train_eval_log_loss evaluates the final weights for that epoch, so it is the directly comparable loss. A growing gap supports overfitting; a high training accuracy alone does not.

In [5]:
overfit_learning_curve = pd.DataFrame(overfit_history)
overfit_learning_curve['accuracy_gap_pp'] = 100 * (
    overfit_learning_curve['train_eval_accuracy'] - overfit_learning_curve['validation_accuracy'])
overfit_learning_curve['validation_minus_train_loss'] = (
    overfit_learning_curve['validation_log_loss'] - overfit_learning_curve['train_eval_log_loss'])
overfit_learning_curve[['epoch', 'train_eval_accuracy', 'validation_accuracy',
    'accuracy_gap_pp', 'train_eval_log_loss', 'validation_log_loss',
    'validation_minus_train_loss']].iloc[::10].copy()

,epoch,train_eval_accuracy,validation_accuracy,accuracy_gap_pp,train_eval_log_loss,validation_log_loss,validation_minus_train_loss
0,1,0.831745,0.8288,0.294483,0.370936,0.379052,0.008116
10,11,0.935766,0.8019,13.386552,0.150109,0.789503,0.639394
20,21,0.970138,0.7963,17.383793,0.071761,1.306107,1.234346
30,31,0.978752,0.7916,18.715172,0.052000,1.470558,1.418558
40,41,0.983041,0.7883,19.474138,0.040972,1.626571,1.585599
50,51,0.983400,0.7888,19.460000,0.040862,1.677509,1.636647
60,61,0.987531,0.7873,20.023103,0.030215,1.755763,1.725548
70,71,0.988579,0.7858,20.277931,0.025897,1.844138,1.818241
80,81,0.987159,0.7927,19.445862,0.030813,1.872293,1.841480
90,91,0.989938,0.7880,20.193793,0.023077,1.961441,1.938364


## Final income-prediction evaluation
Evaluate after the fixed training budget. Do not choose epochs/settings using test results. The membership attack below uses this target and its own fitted preprocessor.


In [6]:
overfit_metrics = {name: evaluate_model(overfit_model, values['X'], values['y'])
    for name, values in overfit_arrays.items() if name in ['train', 'validation', 'test']}
display(pd.DataFrame(overfit_metrics).T)
print('Train-validation accuracy gap (percentage points):',
      100 * (overfit_metrics['train']['accuracy'] - overfit_metrics['validation']['accuracy']))
print('Train-test accuracy gap (percentage points):',
      100 * (overfit_metrics['train']['accuracy'] - overfit_metrics['test']['accuracy']))

,accuracy,roc_auc,log_loss
validation,0.789000,0.854008,1.890478
test,0.791700,0.857344,1.858800
train,0.989166,0.999603,0.025222


Train-validation accuracy gap (percentage points): 20.016551724137933
Train-test accuracy gap (percentage points): 19.74655172413794


## Random forest membership attack: complete-feature baseline
Run after training the overfit 2018 target. This uses the same attack procedure as the overfit notebook: five diverse Adult shadows with original default settings, four paper-inspired features (class-scaled logits, blind walk, entropy, cross-entropy), and a random forest membership classifier.

The attack is fitted on Adult shadow outputs only. Adult attack rows are split by record ID; thresholds are selected on Adult validation only. Feature extraction samples 2,000 members and 2,000 non-members per shadow. Target evaluation uses all 10,000 overfit target MIA members and 10,000 non-members, with this target's own fitted preprocessor. This notebook trains its own attack and reports only its own target result; no cross-notebook results are reused or compared.

Walk defaults: 20 steps, two walks, categorical change probability .05, batch size 512, seed 42. Repeated model queries can take time. The tabular walk is an adaptation, not an exact reproduction; see data/paper_attack_features.md. True income labels are assumed known. No target MIA results select settings. All experiment cells were executed with user authorization for this run.

In [7]:
import importlib
import src.shadow_models
import src.membership_attack
import src.paper_attack_features
importlib.reload(src.shadow_models)
importlib.reload(src.membership_attack)
importlib.reload(src.paper_attack_features)
from src.shadow_models import load_adult_data, train_shadow_models, summarize_shadow_models
from src.membership_attack import train_membership_classifier, evaluate_attack
from src.paper_attack_features import collect_shadow_paper_features, collect_target_paper_features

WALK_CONFIG = dict(max_steps=20, repeats=2, categorical_change_probability=0.05,
                   batch_size=512, seed=42)
SHADOW_RECORDS_PER_CLASS = 2000
# Train the five Adult shadows when you run this cell.
# Skip this cell if shadow_results is already loaded and these imports/settings are set.
adult_auxiliary = load_adult_data()
shadow_results = train_shadow_models(adult_auxiliary, seed=42)
summarize_shadow_models(shadow_results)

Training shadow 1/5: xgboost


Training shadow 2/5: random_forest


Training shadow 3/5: mlp


Epoch 001: train loss=0.4162, validation loss=0.3245, validation accuracy=0.8483


Epoch 002: train loss=0.3166, validation loss=0.3166, validation accuracy=0.8531


Epoch 003: train loss=0.3112, validation loss=0.3153, validation accuracy=0.8525


... (10 outputs left out)

Epoch 014: train loss=0.2939, validation loss=0.3123, validation accuracy=0.8558


Epoch 015: train loss=0.2925, validation loss=0.3146, validation accuracy=0.8551
Training shadow 4/5: decision_tree


Training shadow 5/5: logistic_regression


records  accuracy   roc_auc  log_loss
shadow_model        partition                                        
xgboost             train         19536  0.883241  0.941862  0.254841
                    validation     9769  0.875832  0.927968  0.277922
                    nonmembers    19537  0.871577  0.926679  0.280469
random_forest       train         19536  0.894656  0.958916  0.246325
                    validation     9769  0.865800  0.921603  0.298265
                    nonmembers    19537  0.865281  0.916223  0.305339
mlp                 train         19536  0.862664  0.920766  0.294177
                    validation     9769  0.855154  0.911535  0.311146
                    nonmembers    19537  0.857399  0.913891  0.306424
decision_tree       train         19536  0.879863  0.945487  0.238998
                    validation     9769  0.846351  0.865680  1.032059
                    nonmembers    19537  0.851871  0.871910  0.999966
logistic_regression train         19536  0.856214  0.909753  0.314372
                    validation     9769  0.849729  0.903692  0.322259
                    nonmembers    19537  0.849619  0.903744  0.321650

In [8]:
# Extract the four features by querying this notebook's fitted shadows.
adult_for_rf_queries = load_adult_data()
shadow_attack_data, rf_shadow_walk_diagnostics = collect_shadow_paper_features(
    shadow_results, adult_for_rf_queries, per_class=SHADOW_RECORDS_PER_CLASS,
    config=WALK_CONFIG, seed=42)
display(pd.DataFrame(rf_shadow_walk_diagnostics).T)
attack_result = train_membership_classifier(
    shadow_attack_data, seed=42, algorithm='random_forest', feature_set='paper')
# Adult attack diagnostics, not target membership performance.
pd.DataFrame(attack_result['metrics']).T

Attack feature extraction: 512/4,000


Attack feature extraction: 1,024/4,000


Attack feature extraction: 1,536/4,000


Attack feature extraction: 2,048/4,000


Attack feature extraction: 2,560/4,000


Attack feature extraction: 3,072/4,000


... (32 outputs left out)

Attack feature extraction: 3,584/4,000


Attack feature extraction: 4,000/4,000


,record_queries,max_record_queries,walk_censored_fraction,blind_walk_enabled,config
xgboost,130632,164000,0.685625,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
random_forest,138296,164000,0.769,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
mlp,132415,164000,0.692375,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
decision_tree,117573,164000,0.549625,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
logistic_regression,132848,164000,0.695375,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."


,records,members,accuracy,balanced_accuracy,roc_auc,tpr_at_1pct_fpr_roc,threshold,tn,fp,fn,tp,shadow_selected_threshold,tpr_at_shadow_threshold,fpr_at_shadow_threshold
train,11984.0,5958.0,0.614403,0.614635,0.663945,0.049681,0.5004,3457.0,2569.0,2052.0,3906.0,0.624076,0.020309,0.002323
validation,4023.0,2032.0,0.515536,0.515057,0.510761,0.009350,0.5004,932.0,1059.0,890.0,1142.0,0.624076,0.009350,0.009543
test,3993.0,2010.0,0.517155,0.516845,0.520103,0.004478,0.5004,934.0,1049.0,879.0,1131.0,0.624076,0.006468,0.010590


In [9]:
# Query this notebook's overfit target using its own fitted preprocessor.
target_attack_data, rf_target_walk_diagnostics = collect_target_paper_features(
    overfit_model, overfit_preprocessor, overfit_partitions, config=WALK_CONFIG, seed=42)
display(pd.Series(rf_target_walk_diagnostics, name='target query diagnostics'))
target_attack_metrics, target_membership_scores = evaluate_attack(
    attack_result['classifier'], target_attack_data,
    threshold=attack_result['accuracy_threshold'],
    low_fpr_threshold=attack_result['low_fpr_threshold'])
pd.DataFrame({
    'Adult attack test': attack_result['metrics']['test'],
    'Overfit ACS 2018 target MIA': target_attack_metrics,
}).T

Attack feature extraction: 512/20,000


Attack feature extraction: 1,024/20,000


Attack feature extraction: 1,536/20,000


Attack feature extraction: 2,048/20,000


Attack feature extraction: 2,560/20,000


Attack feature extraction: 3,072/20,000


... (32 outputs left out)

Attack feature extraction: 19,968/20,000


Attack feature extraction: 20,000/20,000


record_queries                                                       533490
max_record_queries                                                   820000
walk_censored_fraction                                               0.4684
blind_walk_enabled                                                     True
config                    {'max_steps': 20, 'repeats': 2, 'categorical_c...
Name: target query diagnostics, dtype: object

,records,members,accuracy,balanced_accuracy,roc_auc,tpr_at_1pct_fpr_roc,threshold,tn,fp,fn,tp,shadow_selected_threshold,tpr_at_shadow_threshold,fpr_at_shadow_threshold
Adult attack test,3993.0,2010.0,0.517155,0.516845,0.520103,0.004478,0.5004,934.0,1049.0,879.0,1131.0,0.624076,0.006468,0.01059
Overfit ACS 2018 target MIA,20000.0,10000.0,0.600150,0.600150,0.613605,0.012500,0.5004,2652.0,7348.0,649.0,9351.0,0.624076,0.075200,0.04780


In [10]:
# Auxiliary records are unused by target training AND all target evaluation splits.
from src.calibrated_attack import run_reference_experiment
used_ids = set().union(*(set(part.index) for part in overfit_partitions.values()))
reference_auxiliary = overfit_source.loc[~overfit_source.index.isin(used_ids)].copy()
assert len(reference_auxiliary) > 10_000
assert set(reference_auxiliary.index).isdisjoint(used_ids)
reference_candidates = pd.concat([
    overfit_partitions['mia_members'].assign(membership=1),
    overfit_partitions['mia_nonmembers'].assign(membership=0),
]).sample(frac=1, random_state=OVERFIT_SEED)
assert reference_candidates.membership.value_counts().to_dict() == {0:10_000,1:10_000}
print('Unused same-year auxiliary pool:', len(reference_auxiliary))
print('Target gap (percentage points):', 100*(overfit_metrics['train']['accuracy']-overfit_metrics['test']['accuracy']))
display(summarize_shadow_models(shadow_results))
print('Baseline attack class counts:')
display(pd.DataFrame({name: frame.membership.value_counts() for name,frame in attack_result['splits'].items()}))


Unused same-year auxiliary pool: 20665
Target gap (percentage points): 19.74655172413794


records  accuracy   roc_auc  log_loss
shadow_model        partition                                        
xgboost             train         19536  0.883241  0.941862  0.254841
                    validation     9769  0.875832  0.927968  0.277922
                    nonmembers    19537  0.871577  0.926679  0.280469
random_forest       train         19536  0.894656  0.958916  0.246325
                    validation     9769  0.865800  0.921603  0.298265
                    nonmembers    19537  0.865281  0.916223  0.305339
mlp                 train         19536  0.862664  0.920766  0.294177
                    validation     9769  0.855154  0.911535  0.311146
                    nonmembers    19537  0.857399  0.913891  0.306424
decision_tree       train         19536  0.879863  0.945487  0.238998
                    validation     9769  0.846351  0.865680  1.032059
                    nonmembers    19537  0.851871  0.871910  0.999966
logistic_regression train         19536  0.856214  0.909753  0.314372
                    validation     9769  0.849729  0.903692  0.322259
                    nonmembers    19537  0.849619  0.903744  0.321650

Baseline attack class counts:


,train,validation,test
membership,,,
0,6026,1991,1983
1,5958,2032,2010


In [11]:
# Train 16 independent reference models; target candidates are OUT of every model.
reference_results = run_reference_experiment(
    reference_auxiliary, reference_candidates, overfit_model, overfit_preprocessor,
    OVERFIT_CONFIG, REPAIR_DIR / 'references', count=16, seed=OVERFIT_SEED)
display(pd.DataFrame(reference_results['reference_diagnostics']))
display(pd.DataFrame(reference_results['attack_partition_balance']))


Reference 1/16: training 10,245 unused ACS records


Epoch 001: train loss=0.5251, validation loss=0.4140, validation accuracy=0.8030


Epoch 002: train loss=0.3967, validation loss=0.4157, validation accuracy=0.8000


Epoch 003: train loss=0.3707, validation loss=0.3972, validation accuracy=0.8200


Epoch 004: train loss=0.3395, validation loss=0.4194, validation accuracy=0.8180


Epoch 005: train loss=0.3012, validation loss=0.4320, validation accuracy=0.8100


... (1,608 outputs left out)

Epoch 099: train loss=0.0027, validation loss=1.8493, validation accuracy=0.8020


Epoch 100: train loss=0.0025, validation loss=1.8703, validation accuracy=0.7960


,reference,train_records,out_records,train_accuracy,out_accuracy,accuracy_gap_pp
0,0,10245,10420,0.999219,0.773129,22.609053
1,1,10292,10373,0.999028,0.766798,23.222995
2,2,10308,10357,0.998060,0.774162,22.389736
3,3,10339,10326,0.996711,0.759248,23.746298
4,4,10438,10227,0.998755,0.771292,22.746287
5,5,10343,10322,0.999227,0.772331,22.689559
6,6,10391,10274,0.998749,0.776231,22.251765
7,7,10264,10401,0.998733,0.775022,22.371180
8,8,10352,10313,0.998648,0.771550,22.709713
9,9,10364,10301,0.998842,0.773905,22.493670


,train,validation,test
1,99192,33064,33064
0,99192,33064,33064


In [2]:
# Report every predeclared method. Target MIA results never choose a winner/configuration.
comparison_rows = [{'method':'original_probability_rf', **target_attack_metrics}]
comparison_rows += [{'method':name, **values['target']}
                    for name,values in reference_results['methods'].items()]
attack_comparison = pd.DataFrame(comparison_rows).set_index('method')
display(attack_comparison[['accuracy','balanced_accuracy','roc_auc','tpr_at_1pct_fpr_roc',
    'tpr_at_shadow_threshold','fpr_at_shadow_threshold']])
summary = dict(target_metrics=overfit_metrics, target_training=overfit_metadata,
    target_train_size=OVERFIT_TRAIN_SIZE, baseline_target_mia=target_attack_metrics,
    baseline_auxiliary_metrics=attack_result['metrics'], reference_results=reference_results,
    caveat='Reference attacks use pre-softmax logits; original RF uses only probabilities. Reference training size differs from the target.')
(REPAIR_DIR / 'summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
attack_comparison.to_csv(REPAIR_DIR / 'attack_comparison.csv')
print('Results saved to',REPAIR_DIR)


,accuracy,balanced_accuracy,roc_auc,tpr_at_1pct_fpr_roc,tpr_at_shadow_threshold,fpr_at_shadow_threshold
method,,,,,,
original_probability_rf,0.60015,0.60015,0.613605,0.0125,0.0752,0.0478
stable_margin_rf,0.60990,0.60990,0.645137,0.0130,0.0089,0.0066
calibrated_rf,0.62050,0.62050,0.683034,0.0601,0.1102,0.0271
stable_margin_threshold,0.60755,0.60755,0.608101,0.0111,0.0527,0.0472
loss_threshold,0.60755,0.60755,0.608101,0.0111,0.0527,0.0472
offline_reference_z,0.60365,0.60365,0.626297,0.0126,0.1597,0.1097


Results saved to artifacts\exp2_attack_repair
